# P18 · U3: GPT-2 small vs from-scratch models + DistilBERT scorer
Runtime ▸ **T4 GPU**, then Run all (about 20 min). Best in the same session as before (needs `data/`).

## Setup and data

In [ ]:
import os, sys, re, time, math, json, random, copy, zipfile, shutil, glob, subprocess
import torch, torch.nn.functional as F
import pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

assert torch.cuda.is_available(), "Switch runtime to T4 GPU: Runtime > Change runtime type"
dev = "cuda"
for d in ("src", "out", "data"): os.makedirs(d, exist_ok=True)
if not os.path.exists("src/model.py"):
    from google.colab import files
    files.upload()                       # choose p18_repo.zip
    zipfile.ZipFile("p18_repo.zip").extractall(".")
    for f in glob.glob("p18repo/src/*.py"): shutil.copy(f, "src/")
    for f in glob.glob("p18repo/out/*"): shutil.copy(f, "out/")
if not os.path.exists("data/train.txt"):
    print("data/ missing (session reset): rebuilding splits. GPT-2 is trained and tested on this NEW split; "
          "mention in the report that the char models used an earlier split of the same data.")
    subprocess.run([sys.executable, "src/data.py"], check=True)
load = lambda p: [l for l in open(p).read().split("\n") if l]
tr, va, te = load("data/train.txt"), load("data/val.txt"), load("data/test.txt")
known = set(tr)
print(len(tr), len(va), len(te))

## Fine-tune GPT-2 small on the same names (2 epochs)

In [ ]:
tok = AutoTokenizer.from_pretrained("gpt2"); EOS = tok.eos_token_id      # EOS is also the start token

def encode_names(names):
    enc = tok(names, add_special_tokens=False)["input_ids"]
    return [[EOS] + ids + [EOS] for ids in enc]

def batchify(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), EOS); mask = torch.zeros(len(seqs), L, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s); mask[i, :len(s)] = 1
    return ids.to(dev), mask.to(dev)

def nll_sum(model, ids, mask):
    logits = model(input_ids=ids, attention_mask=mask).logits[:, :-1]
    tgt = ids[:, 1:]; m = mask[:, 1:].bool()
    return F.cross_entropy(logits[m].float(), tgt[m], reduction="sum")

@torch.no_grad()
def eval_bpc(model, names, bs=512):
    """Bits per CHARACTER (name chars + 1 end symbol), same normalisation as the char-level models."""
    model.eval(); seqs = encode_names(names); tot = 0.0
    for i in range(0, len(seqs), bs):
        ids, mask = batchify(seqs[i:i+bs])
        with torch.autocast("cuda", dtype=torch.float16):
            tot += nll_sum(model, ids, mask).item()
    model.train()
    return tot / sum(len(n) + 1 for n in names) / math.log(2)

model = AutoModelForCausalLM.from_pretrained("gpt2").to(dev)
n_params = sum(p.numel() for p in model.parameters())
print(f"GPT-2 params: {n_params:,}")
epochs, bs, lr, warm = 2, 256, 1e-4, 100
seqs = encode_names(tr)
total = epochs * math.ceil(len(seqs) / bs)
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")
sched = lambda s: lr * (s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
steps, tr_b, va_b, step, run, t0 = [], [], [], 0, 0.0, time.time()
val_small = va[:5000]
model.train()
for ep in range(epochs):
    perm = torch.randperm(len(seqs)).tolist()
    for i in range(0, len(perm), bs):
        idx = perm[i:i+bs]
        ids, mask = batchify([seqs[j] for j in idx])
        chars = sum(len(tr[j]) + 1 for j in idx)
        for g in opt.param_groups: g["lr"] = sched(step)
        with torch.autocast("cuda", dtype=torch.float16):
            loss_sum = nll_sum(model, ids, mask)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss_sum / chars).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        b = loss_sum.item() / chars / math.log(2)
        run = 0.9 * run + 0.1 * b if step else b; step += 1
        if step % 300 == 0 or step == total:
            v = eval_bpc(model, val_small)
            steps.append(step); tr_b.append(run); va_b.append(v)
            print(f"ep {ep+1} step {step}/{total} train {run:.3f} bpc  val {v:.3f} bpc")
train_secs = time.time() - t0
gpt_val, gpt_test = eval_bpc(model, va), eval_bpc(model, te)
print(f"GPT-2 fine-tuned: val {gpt_val:.3f} bpc, test {gpt_test:.3f} bpc, {train_secs:.0f}s")
plt.figure(figsize=(6, 4)); plt.plot(steps, tr_b, label="train (smoothed)"); plt.plot(steps, va_b, label="validation")
plt.xlabel("step"); plt.ylabel("bits per character"); plt.title("GPT-2 small fine-tuning loss")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("out/gpt2_loss_curve.png", dpi=150)

## DistilBERT real-vs-corrupted scorer (Layer III: BERT-family model)

In [ ]:
# DistilBERT plausibility scorer: real place names vs corrupted names (shuffled / random / mutated letters).
# Corruptions do not come from any generator, so the score is fair to every model.
bt = AutoTokenizer.from_pretrained("distilbert-base-uncased")
rng = random.Random(0)
POOL = "".join(tr[:20000]).replace(" ", "")          # letter pool with real frequencies
def corrupt(n):
    k = rng.choice(["shuffle", "random", "mutate"])
    if k == "shuffle":
        c = list(n.replace(" ", "")); rng.shuffle(c); return "".join(c)
    if k == "random":
        return "".join(rng.choice(POOL) for _ in range(len(n)))
    c = list(n)
    for i in range(len(c)):
        if c[i] != " " and rng.random() < 0.3: c[i] = rng.choice(POOL)
    return "".join(c)

real_tr = rng.sample(tr, 20000); fake_tr = [corrupt(n) for n in rng.sample(tr, 20000)]
X = real_tr + fake_tr; Y = [1] * len(real_tr) + [0] * len(fake_tr)
real_te = te[:2000]; fake_te = [corrupt(n) for n in te[2000:4000]]
clf = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(dev)
opt_c = torch.optim.AdamW(clf.parameters(), lr=5e-5); sc = torch.amp.GradScaler("cuda")
order = list(range(len(X))); rng.shuffle(order); clf.train()
for i in range(0, len(order), 64):
    b = order[i:i+64]
    enc = bt([X[j] for j in b], padding=True, truncation=True, max_length=24, return_tensors="pt").to(dev)
    y = torch.tensor([Y[j] for j in b], device=dev)
    with torch.autocast("cuda", dtype=torch.float16):
        loss = F.cross_entropy(clf(**enc).logits.float(), y)
    opt_c.zero_grad(); sc.scale(loss).backward(); sc.step(opt_c); sc.update()

@torch.no_grad()
def probs(names, bs=256):
    clf.eval(); out = []
    for i in range(0, len(names), bs):
        enc = bt([n.lower() for n in names[i:i+bs]], padding=True, truncation=True, max_length=24, return_tensors="pt").to(dev)
        with torch.autocast("cuda", dtype=torch.float16):
            lg = clf(**enc).logits
        out += torch.softmax(lg.float(), -1)[:, 1].tolist()
    return out
pr, pf = probs(real_te), probs(fake_te)
acc = (sum(p > .5 for p in pr) + sum(p <= .5 for p in pf)) / (len(pr) + len(pf))
print(f"DistilBERT held-out accuracy (real vs corrupted): {acc:.3f} | mean P(real): real {sum(pr)/len(pr):.3f}, corrupted {sum(pf)/len(pf):.3f}")
json.dump({"bert_heldout_accuracy": acc, "mean_p_real_real": sum(pr)/len(pr), "mean_p_real_corrupted": sum(pf)/len(pf)},
          open("out/bert_scorer_metrics.json", "w"), indent=2)

## Prompting vs fine-tuning, and the final table

In [ ]:
@torch.no_grad()
def gen_ft(m, n=1000, T=0.8, seed=0):
    torch.manual_seed(seed); d = next(m.parameters()).device
    inp = torch.full((n, 1), EOS, device=d)
    out = m.generate(inp, attention_mask=torch.ones_like(inp), do_sample=True, temperature=T, top_k=0,
                     max_new_tokens=16, pad_token_id=EOS, eos_token_id=EOS)
    names = []
    for row in out.tolist():
        ids = row[1:]
        if EOS in ids: ids = ids[:ids.index(EOS)]
        names.append(tok.decode(ids).strip())
    return [x for x in names if x]

base = AutoModelForCausalLM.from_pretrained("gpt2").to(dev).eval()     # NOT fine-tuned
@torch.no_grad()
def prompt_gen(prompt, n=200, T=0.8, seed=0):
    torch.manual_seed(seed)
    enc = tok(prompt, return_tensors="pt").to(dev)
    out = base.generate(**enc, do_sample=True, temperature=T, top_k=0, max_new_tokens=8,
                        num_return_sequences=n, pad_token_id=EOS)
    res = []
    for row in out.tolist():
        line = tok.decode(row[enc["input_ids"].shape[1]:]).strip().split("\n")[0]
        res.append(re.sub(r"[^A-Za-z ]", "", line).strip().lower())
    return [r for r in res if 3 <= len(r) <= 24]

zero = prompt_gen("The name of a small village in India:")
shots = "\n".join(n.title() for n in random.Random(1).sample(tr, 8))
few = prompt_gen("Indian village names:\n" + shots + "\n")
ft = gen_ft(model)
sets = {"GPT-2 zero-shot prompt": zero, "GPT-2 few-shot prompt": few, "GPT-2 fine-tuned": ft}

if os.path.exists("out/mini_gpt.pt"):
    sys.path.insert(0, "src")
    from sample import load_model, generate
    tm, ts, ti = load_model("out/mini_gpt.pt")
    sets["Mini-Transformer (ref)"] = [n.lower() for n in generate(tm, ts, ti, n=1000, temperature=0.8, seed=0) if n]

def stats(names):
    names = [n.lower() for n in names]; u = set(names)
    return {"valid_names": len(names), "unique_%": 100 * len(u) / max(1, len(names)),
            "novel_%": 100 * len([n for n in u if n not in known]) / max(1, len(u)),
            "plausibility (BERT P(real))": sum(probs(names)) / max(1, len(names))}
rows = {k: stats(v) for k, v in sets.items()}
rows["Reference: real test names"] = {"plausibility (BERT P(real))": sum(pr) / len(pr)}
rows["Reference: corrupted names"] = {"plausibility (BERT P(real))": sum(pf) / len(pf)}

cpu = copy.deepcopy(model).cpu().float().eval(); t = time.time()
for k in range(20): gen_ft(cpu, n=1, seed=k)
rows["GPT-2 fine-tuned"].update({"params": n_params, "val_bpc": gpt_val, "test_bpc": gpt_test,
                                 "train_s": train_secs, "cpu_ms_per_name": (time.time() - t) / 20 * 1000})
rows["GPT-2 zero-shot prompt"]["params"] = rows["GPT-2 few-shot prompt"]["params"] = n_params
table = pd.DataFrame(rows).T.round(3)
table.to_csv("out/gpt2_bert_table.csv"); print(table.to_string())
for k, v in sets.items(): print(f"\n{k}:", [n.title() for n in v[:10]])

## Download

In [ ]:
!zip -j gpt2_results.zip out/gpt2_loss_curve.png out/gpt2_bert_table.csv out/bert_scorer_metrics.json
from google.colab import files; files.download("gpt2_results.zip")